# Chapter 1 &mdash; Context-Sensitive Patterns: a Copy, Not a Reversal

**Concept 12 of the Chapter 1 decomposition:** *Pattern Class III -- Context-Sensitive Patterns*

A function prototype must match its definition. Abstractly that is $ww$ &mdash; and a stack cannot do it, because a stack hands things back <b>reversed</b>.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Context-Sensitive-Patterns/Concept-Context-Sensitive-Patterns.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.LangDef        import *
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_TM         import *
from jove.AnimateTM      import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


`char func(int, float);` must agree with `char func(int a, float b) { body }` &mdash; the same
list of types, **in the same order**.

Abstracted to bits, that is $ww$: a pattern followed by a **copy**, with no reversal.

* $ww^R$ &mdash; palindrome &mdash; **context-free**. A stack gives you reversal for free.
* $ww$ &nbsp;&nbsp; &mdash; copy &nbsp;&nbsp;&mdash; **not** context-free. A stack cannot replay in order.

This is the sharpest single illustration of why the hierarchy is strict.

## 2. Definitions

### Why a stack fails on $ww$

Push all of $w$, then pop: you get $w$ **backwards**. That is exactly wrong for a copy
and exactly right for a palindrome.

In [ ]:
def stack_replay(w):
    stack = []
    for ch in w:
        stack.append(ch)          # push
    out = ''
    while stack:
        out += stack.pop()        # pop -- comes back reversed
    return out

w = '0110100'
print("w              =", w)
print("stack replay   =", stack_replay(w), " <- reversed, so it matches w w^R")
print("but we need    =", w,               " <- a COPY, for w w")

### The prototype/definition problem, in miniature

In [ ]:
def types_of(decl):
    inside = decl[decl.index('(')+1:decl.index(')')]
    return [t.strip().split()[0] for t in inside.split(',') if t.strip()]

def prototype_matches(proto, defn):
    return types_of(proto) == types_of(defn)

### A Turing machine for $w\#w$

With a separator the job becomes deterministic: mark a symbol on the left, walk right
past `#`, check and mark the matching one, walk back. A TM can do this because it can
**revisit** the tape &mdash; which a stack cannot.

In [ ]:
wpw = md2mc('''TM
I     : 0 ; X , R -> Go0
I     : 1 ; Y , R -> Go1
I     : # ; # , R -> ChkE
Go0   : 0 ; 0 , R -> Go0
Go0   : 1 ; 1 , R -> Go0
Go0   : # ; # , R -> Sk0
Sk0   : X ; X , R -> Sk0
Sk0   : Y ; Y , R -> Sk0
Sk0   : 0 ; X , L -> Back
Go1   : 0 ; 0 , R -> Go1
Go1   : 1 ; 1 , R -> Go1
Go1   : # ; # , R -> Sk1
Sk1   : X ; X , R -> Sk1
Sk1   : Y ; Y , R -> Sk1
Sk1   : 1 ; Y , L -> Back
Back  : 0 ; 0 , L -> Back
Back  : 1 ; 1 , L -> Back
Back  : X ; X , L -> Back
Back  : Y ; Y , L -> Back
Back  : # ; # , L -> Back2
Back2 : 0 ; 0 , L -> Back2
Back2 : 1 ; 1 , L -> Back2
Back2 : X ; X , R -> I
Back2 : Y ; Y , R -> I
Back2 : . ; . , R -> I
ChkE  : X ; X , R -> ChkE
ChkE  : Y ; Y , R -> ChkE
ChkE  : . ; . , S -> Fin
''')
print("w#w TM states :", len(wpw["Q"]))

## 3. Tests

Prototype matching, on the book's example.

In [ ]:
proto = "char func(int, float);"
good  = "char func(int a, float b) { body }"
bad   = "char func(float a, int b) { body }"
print("types in prototype :", types_of(proto))
assert types_of(proto) == ["int", "float"]
print("matches good defn  :", prototype_matches(proto, good))
print("matches bad  defn  :", prototype_matches(proto, bad), " <- order matters")
assert prototype_matches(proto, good) and not prototype_matches(proto, bad)

Run the TM on $w\#w$. It accepts a genuine copy and rejects a mismatch.

In [ ]:
explore_tm(wpw, "001#001", 200)

Where each pattern class lives.

In [ ]:
print("REGULAR          : a b a b a b ...      no memory of how many")
print("CONTEXT-FREE     : w w^R  (palindrome)  a stack, read back reversed")
print("CONTEXT-SENSITIVE: w w    (a copy)      needs to re-read -- LBA / TM")

## 4. Animation


Step the $w\#w$ machine. Watch it mark a symbol (`0` becomes `X`), cross the `#`,
find the partner, and walk back. **Re-reading the tape is the power a stack lacks.**


*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateTM import *
AnimateTM(wpw, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Run `explore_tm(wpw, "01#10", 200)`. It should reject. Trace *where* it gets stuck.
2. Why is $ww$ harder than $w\#w$? (Hint: without `#` you must *guess* the midpoint &mdash;
   Chapter 13 builds a nondeterministic TM for exactly this.)
3. Give a real programming-language rule, other than prototypes, that is
   context-sensitive.

In [ ]:
# Your work for the exercises above.